In [2]:
import time
from functools import wraps

In [7]:
def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter() - start
        print (end)
        return result
    return wrapper

In [8]:
@timer
def add(a, b):
    """Return sum of two numbers."""
    return a + b

In [10]:
assert add(2,3)==5

1.500011421740055e-06


In [11]:
assert add.__name__=='add'

In [12]:
assert add.__doc__ == "Return sum of two numbers."

In [22]:
def timer_repeat(runs=5):
    def timer(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            mid = 0
            for _ in range(runs):
                start = time.perf_counter()
                result = func(*args, **kwargs)
                end = time.perf_counter() - start
                mid+=end
            print(mid/runs)
            return result
        
        return wrapper
    return timer

In [23]:
@timer_repeat(runs=5)
def add(a, b):
    """Return sum of two numbers."""
    return a + b

In [24]:
add(2,3)

5.400041118264199e-07


5

In [30]:
def retry(retries, delay, exceptions):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(retries+1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exception=e
                    if attempt < retries:
                        time.sleep(delay)
            raise last_exception
                    
        return wrapper
    return decorator

In [53]:
calls = {"count": 0}

@retry(retries=2, delay=0, exceptions=(ValueError,))
def always_fails():
    calls["count"] += 1
    raise ValueError("still broken")

In [54]:

try:
    always_fails()
except ValueError as e:
    assert str(e) == "still broken"
else:
    assert False, "ValueError was not raised"

assert calls["count"] == 3

In [34]:
@retry(retries=5, delay=0, exceptions=(ValueError,))
def raises_type_error():
    raise TypeError("wrong type")

In [35]:
try:
    raises_type_error()
except TypeError as e:
    assert str(e) == "wrong type"
else:
    assert False, "TypeError was not raised"

In [55]:
from pathlib import Path
import json
import csv

def read_file(path):
    path = Path(path)
    if path.suffix == '.json':
        with path.open('r', encoding='utf-8') as file:
            return json.load(file)
        
    elif path.suffix=='.csv':
        with path.open('r', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            return (list(reader))
    else:
        raise ValueError(f'Unsupported file format: {path.suffix}')
    

In [56]:
assert read_file('data/users.json') == [
    {"name": "Anna", "age": 22},
    {"name": "Boris", "age": 25},
]

In [57]:
assert read_file("data/users.csv") == [
    {"name": "Anna", "age": "22"},
    {"name": "Boris", "age": "25"},
]

In [58]:
try:
    read_file('data/broken.json')
except json.JSONDecodeError:
    pass
else:
    assert False, 'json.JSONDecodeError was not raised'

In [62]:
try:
    read_file('data/users.csv')
except FileNotFoundError:
    pass
else:
    assert False, 'FileNotFound not raised'

AssertionError: FileNotFound not raised

In [60]:
try:
    read_file('data/users.txt')
except ValueError as e:
    assert str(e) == "Unsupported file format: .txt"
else:
    assert False, 'ValueError was not raised'